# Étape 2 : Développement des Modèles

**Trois approches mathématiques distinctes :**
1. **Régression Logistique** avec pénalité Elastic Net (baseline)
2. **Random Forest** avec Analyse de Proximité et détection d'outliers
3. **XGBoost** Cost-Sensitive + Optimisation Bayésienne (Optuna)

> **Rappel** : Ne pas utiliser l'Accuracy comme métrique principale (déceptive à 1:150).

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), 'src') if os.path.basename(os.getcwd()) == 'notebooks' else os.path.join(os.getcwd(), 'src'))

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from IPython.display import display, Image

from config import *
from eda import run_eda

print('Chargement et préparation des données...')
eda_data = run_eda(OUTPUT_DIR)

X_train = eda_data['X_train']
X_val   = eda_data['X_val']
X_test  = eda_data['X_test']
y_train = eda_data['y_train']
y_val   = eda_data['y_val']
y_test  = eda_data['y_test']
feature_cols = eda_data['feature_cols']

print(f'\nDonnées prêtes : X_train={X_train.shape}, X_test={X_test.shape}')

## 2.1 Modèle 1 — Régression Logistique (Elastic Net)

### Justification des hyperparamètres

**Elastic Net** combine L1 (Lasso) + L2 (Ridge) :
$$\text{pénalité} = \alpha \cdot l_1\_ratio \cdot |w| + \frac{\alpha}{2} \cdot (1 - l_1\_ratio) \cdot \|w\|^2$$

- **L1** → sélection de features par sparsité (coefficients nuls pour features peu utiles)
- **L2** → stabilité face à la multicolinéarité des features PCA
- **C** : force de régularisation, optimisé par CV-AUPRC sur 5 folds
- **class_weight='balanced'** : poids inversement proportionnels aux fréquences (1:150 → fraude reçoit 150× plus de poids)
- **solver='saga'** : seul solver compatible avec Elastic Net

In [ ]:
from model_logistic import train_logistic_regression

lr_results = train_logistic_regression(
    X_train, y_train, X_val, y_val, X_test, y_test, OUTPUT_DIR
)

print(f'\nMeilleurs hyperparamètres : {lr_results["best_params"]}')
print(f'ECE (calibration)         : {lr_results["ece"]:.4f}')

In [ ]:
print('Coefficients ElasticNet (top features) :')
display(Image(filename=os.path.join(OUTPUT_DIR, 'logistic_coefficients.png')))

## 2.2 Modèle 2 — Random Forest avec Analyse de Proximité

### Justification des hyperparamètres

| Hyperparamètre | Valeur | Justification |
|---|---|---|
| `n_estimators` | 300 | Stabilité des estimations de proximité (>200 arbres) |
| `max_depth` | 20 | Capture les patterns frauduleux complexes |
| `min_samples_leaf` | 5 | Feuilles suffisamment peuplées pour proximité significative |
| `class_weight` | balanced | Compensation automatique du ratio 1:150 |
| `max_features` | sqrt | Heuristique RF standard, réduit la corrélation inter-arbres |

### Matrice de proximité
$$P[i,j] = \frac{\text{nombre d'arbres où } i \text{ et } j \text{ finissent dans la même feuille}}{\text{nombre total d'arbres}}$$

### Score d'outlier de prédiction
$$\text{outlier\_score}(i) = \frac{n}{\sum_{j \in \text{classe}(i)} P[i,j]^2}$$
Score élevé → observation isolée de ses voisins de classe → le modèle hésite

In [ ]:
from model_rf import train_random_forest

rf_results = train_random_forest(
    X_train, y_train, X_val, y_val, X_test, y_test, OUTPUT_DIR
)

print(f'\nOOB Score : {rf_results["model"].oob_score_:.4f}')
print(f'ECE       : {rf_results["ece"]:.4f}')

In [ ]:
print('Feature importances RF :')
display(Image(filename=os.path.join(OUTPUT_DIR, 'rf_feature_importance.png')))

print('\nAnalyse des outliers de prédiction (t-SNE + matrice de proximité) :')
display(Image(filename=os.path.join(OUTPUT_DIR, 'rf_proximity_outliers.png')))

In [ ]:
# Lire le rapport textuel sur les outliers
with open(os.path.join(OUTPUT_DIR, 'rf_outlier_explanation.txt')) as f:
    print(f.read())

## 2.3 Modèle 3 — XGBoost Cost-Sensitive + Optimisation Bayésienne

### Deux stratégies cost-sensitive comparées

**Stratégie A — scale_pos_weight** :
$$\text{scale\_pos\_weight} = \frac{N_{\text{normal}}}{N_{\text{fraude}}} \approx 578$$
Amplifie les mises à jour de gradient pour la classe minoritaire.

**Stratégie B — Focal Loss** :
$$\text{FL}(p_t) = -\alpha \cdot (1 - p_t)^\gamma \cdot \log(p_t)$$
- $(1-p_t)^\gamma$ : down-weighting des exemples faciles (transactions normales bien classifiées)
- $\gamma=2$ : standard (Lin et al., 2017) — les exemples avec $p>0.9$ reçoivent $0.1^2=1\%$ du poids
- $\alpha=0.75$ : poids supplémentaire pour la classe fraude

### Justification de l'espace de recherche Optuna (TPE)

| Hyperparamètre | Plage | Justification |
|---|---|---|
| `max_depth` | [3, 10] | Profondeur 3=généralisation, 10=capacité max; optimal ~4-7 sur données tabulaires |
| `learning_rate` | [0.01, 0.3] log | Log-uniforme: plus de budget sur les faibles lr (meilleures généralisation) |
| `n_estimators` | [100, 600] | Plus d'arbres compense un lr faible |
| `subsample` | [0.6, 1.0] | Sous-échantillonnage des lignes réduit la variance |
| `colsample_bytree` | [0.6, 1.0] | Sous-échantillonnage des features (comme Random Forest) |
| `gamma` | [0, 5] | Seuil min de réduction de perte pour un split = régularisation |
| `reg_lambda` | [1, 10] | L2 sur poids des feuilles: évite les grandes mises à jour |
| `reg_alpha` | [0, 5] | L1: sparsité dans les poids |
| `min_child_weight` | [1, 10] | Somme min de hessien dans une feuille: contrôle overfitting |

In [ ]:
from model_xgboost import train_xgboost

print(f'Nombre de trials Optuna : {N_TRIALS} par stratégie')
print(f'Timeout                 : {OPTUNA_TIMEOUT}s par stratégie')
print(f'Métrique optimisée      : AUPRC (Average Precision)\n')

xgb_results = train_xgboost(
    X_train, y_train, X_val, y_val, X_test, y_test, OUTPUT_DIR
)

print(f'\nStratégie gagnante : {xgb_results["best_strategy"]}')
print(f'ECE               : {xgb_results["ece"]:.4f}')

In [ ]:
print('Convergence Optuna (Optimization History) :')
display(Image(filename=os.path.join(OUTPUT_DIR, 'optuna_convergence.png')))

print('\nComparaison des deux stratégies cost-sensitive :')
display(Image(filename=os.path.join(OUTPUT_DIR, 'xgb_strategy_comparison.png')))

print('\nFeature importances XGBoost (Gain) :')
display(Image(filename=os.path.join(OUTPUT_DIR, 'xgb_feature_importance.png')))

## Résumé de l'étape 2

| Modèle | Approche imbalance | Optimisation |
|--------|-------------------|---------------|
| Logistic Regression | class_weight='balanced' | Grid Search 5-fold CV AUPRC |
| Random Forest | class_weight='balanced' | Paramètres justifiés théoriquement |
| XGBoost SPW | scale_pos_weight | Optuna TPE (50 trials) |
| XGBoost Focal | Focal Loss custom | Optuna TPE (50 trials) |